# Corporate Financial Distress Classification

Qidian (Martin) Xing. An owner-directed, Codex-assisted extension of the earlier team project. Team credit and primary sources remain in the project acknowledgements.

This notebook reports actual completed experiments. The 2015-2018 period was inspected in earlier versions; it is not a fresh holdout. P2 ranks retrospective company status and does not establish one-year default probabilities. Final baselines and selected recipes share the 2014 fitting cutoff. Search targets, failures and budget shortfalls remain explicit.


In [1]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score
from IPython.display import display

ROOT=Path.cwd()
if not (ROOT/'configs/protocol.yaml').exists():
    ROOT=ROOT.parent
assert (ROOT/'configs/protocol.yaml').exists()
for directory in [ROOT/'src']:
    sys.path.insert(0,str(directory))
saved_verification=json.loads((ROOT/'results/verification.json').read_text())
assert saved_verification['passed']
selected=json.loads((ROOT/'configs/selection_freeze.json').read_text())
table=pd.read_csv(ROOT/'results/historical_2015_2018.csv').set_index('name')
print('Saved final verification passed; private-artifact checks below run separately.')
display(pd.DataFrame([{
    'protected_files':saved_verification['protected_files_unchanged'],
    'prediction_sets':saved_verification['saved_prediction_sets_verified'],
    'development_sets':saved_verification['development_prediction_sets_verified'],
    'authorized_windows':saved_verification['authorized_time_windows_verified']
}]))


Saved final verification passed; private-artifact checks below run separately.


,protected_files,prediction_sets,development_sets,authorized_windows
0,1344,47,541,2


## Same-cutoff comparison

Annual AP averages the four historical years equally. Pooled AP and ROC use combined years. Recall@5% applies the review budget separately each year. Averaging predictions across seeds differs from averaging seed-level metrics. Archived and strict P1 rows retain their separate roles in the complete comparison CSV.


In [2]:
baseline_keys=['baseline_legacy_strong_inner_l1','baseline_v1_lags_xgb',
               'baseline_v1_raw_catboost','baseline_v1_selected_rf']
keys=list(dict.fromkeys([selected['overall'],selected['DL'],*baseline_keys]))
names=['P2_seed_mean__'+key for key in keys]
primary_name='P2_seed_mean__'+selected['overall']
if selected['blend']['admitted']:
    primary_name='P2_fixed_rank_blend'
    names=[primary_name]+['P2_annual_rank__'+key for key in keys]+names
display(table.loc[names,['candidate','channel','fit_cutoff','macro_AP','AP','ROC_AUC','recall5']].round(6))


,candidate,channel,fit_cutoff,macro_AP,AP,ROC_AUC,recall5
name,,,,,,,
P2_fixed_rank_blend,frozen_blend,P2_annual_rank_seed_ensemble,2014,0.373320,0.382691,0.921602,0.619824
P2_annual_rank__refine_r3_15_a028aa7a,refine_r3_15_a028aa7a,P2_annual_rank_seed_ensemble,2014,0.312815,0.310077,0.904828,0.540400
P2_annual_rank__refine_r3_21_bbf84fb1,refine_r3_21_bbf84fb1,P2_annual_rank_seed_ensemble,2014,0.392916,0.388584,0.913006,0.621462
P2_annual_rank__baseline_legacy_strong_inner_l1,baseline_legacy_strong_inner_l1,P2_annual_rank_seed_ensemble,2014,0.262098,0.254136,0.860843,0.524208
P2_annual_rank__baseline_v1_lags_xgb,baseline_v1_lags_xgb,P2_annual_rank_seed_ensemble,2014,0.302835,0.299541,0.852568,0.513465
P2_annual_rank__baseline_v1_raw_catboost,baseline_v1_raw_catboost,P2_annual_rank_seed_ensemble,2014,0.279037,0.270171,0.851065,0.487553
P2_annual_rank__baseline_v1_selected_rf,baseline_v1_selected_rf,P2_annual_rank_seed_ensemble,2014,0.286383,0.281000,0.892026,0.503744
P2_seed_mean__refine_r3_15_a028aa7a,refine_r3_15_a028aa7a,P2_seed_ensemble,2014,0.312815,0.324517,0.903908,0.540400
P2_seed_mean__refine_r3_21_bbf84fb1,refine_r3_21_bbf84fb1,P2_seed_ensemble,2014,0.392916,0.391599,0.912470,0.621462


## Independent arithmetic from saved predictions

Private company-level predictions are intentionally excluded from public Git. When those local files are present, this cell checks annual/pooled metrics and the exact public results. Its outputs contain only aggregates. A public clone without private artifacts still displays the released comparison.


In [3]:
from distress_ml.splits import tie_key
prediction_name=('P2_fixed_rank_blend_historical.csv.gz' if selected['blend']['admitted']
                 else 'P2_seed_mean__'+selected['overall']+'_historical.csv.gz')
prediction_path=ROOT/'artifacts/predictions'/prediction_name
if prediction_path.exists():
    prediction=pd.read_csv(prediction_path,float_precision='round_trip')
    assert prediction.row_id.is_unique
    annual=[]
    for year,g in prediction.groupby('year'):
        y=g.y_true.to_numpy(int);q=g.score.to_numpy()
        order=np.lexsort(([tie_key(company,int(year)) for company in g.company_id],-q))
        count=int(np.ceil(.05*len(g)))
        annual.append(dict(year=int(year),rows=len(g),positives=int(y.sum()),
                           AP=average_precision_score(y,q),ROC_AUC=roc_auc_score(y,q),
                           recall5=float(y[order[:count]].sum()/y.sum())))
    annual=pd.DataFrame(annual)
    reference=table.loc[primary_name]
    np.testing.assert_allclose(annual.AP.mean(),reference.macro_AP,atol=1e-9)
    np.testing.assert_allclose(annual.recall5.mean(),reference.recall5,atol=1e-9)
    np.testing.assert_allclose(average_precision_score(prediction.y_true,prediction.score),
                               reference.AP,atol=1e-9)
    np.testing.assert_allclose(roc_auc_score(prediction.y_true,prediction.score),
                               reference.ROC_AUC,atol=1e-9)
    released=pd.read_csv(ROOT/'results/historical_by_year.csv')
    released=released[released.name==primary_name].sort_values('year')
    np.testing.assert_allclose(annual[['AP','ROC_AUC','recall5']],
                               released[['AP','ROC_AUC','recall5']],atol=1e-9)
    display(annual.round(6))
    print('Independent annual and pooled arithmetic passed.')
else:
    print('Private predictions are absent. No independent local arithmetic is claimed.')


Private predictions are absent. No independent local arithmetic is claimed.


## Reload one selected tree and one selected neural pipeline

Only trusted local artifacts generated by this project are loaded. Frozen hashes, source-row alignment and saved raw scores are checked. This cell performs inference only and cannot reopen model selection.


In [4]:
model_root=ROOT/'artifacts/models'
if model_root.exists() and (ROOT/'artifacts/historical_complete.json').exists():
    import hashlib
    from distress_v2.training import load as load_pipeline
    from distress_v2.data import load as load_data, features
    freeze=json.loads((ROOT/'configs/artifact_freeze.json').read_text())
    data=load_data('historical');mask=data.year>=2015
    reloaded=[]
    for role in ['ML','DL']:
        item=next(m for m in freeze['models'] if m['channel']=='P2'
                  and m['candidate']==selected[role] and m['seed']==42)
        path=ROOT/item['path']
        assert hashlib.sha256(path.read_bytes()).hexdigest()==freeze['artifacts'][item['path']]
        fitted=load_pipeline(path)
        score=fitted.predict(features(data,fitted.cfg['features']).loc[mask])
        saved=pd.read_csv(ROOT/'artifacts/predictions'/
                          (item['name']+'_historical.csv.gz'),float_precision='round_trip')
        np.testing.assert_array_equal(data.loc[mask,'row_id'],saved.row_id)
        np.testing.assert_allclose(score,saved.raw_score,atol=1e-6,rtol=1e-6)
        reloaded.append(dict(role=role,family=fitted.cfg['family'],seed=42,
                             fit_cutoff=2014,rows=len(score),
                             maximum_reload_error=float(np.max(abs(score-saved.raw_score)))))
    display(pd.DataFrame(reloaded))
    print('Selected tree/neural pipeline reload checks passed.')
else:
    print('Trusted private models are absent. No model reload is claimed.')


Trusted private models are absent. No model reload is claimed.


## Search coverage, uncertainty and company stress

Completed configurations are distinct from folds, seeds and TabM heads. Bootstrap intervals condition on fitted models and the current sample, omit full HPO/macro uncertainty, and retain paired company draws. Annual metrics require the same fixed year set. Company group tests use hyperparameters selected on broader companies and are post-selection stress tests.


In [5]:
trials=pd.read_csv(ROOT/'results/trials.csv')
display(trials.groupby(['stage','family','status']).size().rename('configurations').unstack(fill_value=0))
target='fixed_annual_rank_blend' if selected['blend']['admitted'] else selected['overall']
intervals=pd.read_csv(ROOT/'results/paired_company_intervals.csv')
display(intervals[(intervals.model==target)&intervals.metric.isin(
    ['macro_year_AP','macro_year_recall_top5'])][
    ['reference','metric','delta_mean','delta_lower','delta_upper','valid_draws']].round(6))
display(pd.read_csv(ROOT/'results/company_stress_summary.csv').round(6))
print('Company stress ranges are not confidence intervals.')


status                      COMPLETE  INCOMPLETE_BUDGET  \
stage      family                                         
baseline   catboost                1                  0   
           legacy_residual         1                  0   
           logistic                1                  0   
           random_forest           1                  0   
           v1_residual             1                  0   
           xgboost                 1                  0   
bridge     legacy_residual        15                  0   
           xgboost                 4                  0   
refinement catboost                4                  0   
           mlp                    11                  0   
           xgboost                15                  1   
screen     catboost               36                  0   
           extra_trees             6                  0   
           legacy_residual         6                  1   
           mlp                     6                  0   
           num_mlp                 6                  0   
           random_forest           6                  1   
           tabm                    6                  0   
           xgboost                36                  0   

status                      INVALIDATED_STEP_CONTRACT  
stage      family                                      
baseline   catboost                                 0  
           legacy_residual                          0  
           logistic                                 0  
           random_forest                            0  
           v1_residual                              1  
           xgboost                                  0  
bridge     legacy_residual                          0  
           xgboost                                  0  
refinement catboost                                 0  
           mlp                                      0  
           xgboost                                  0  
screen     catboost                                 0  
           extra_trees                              0  
           legacy_residual                          0  
           mlp                                      0  
           num_mlp                                  0  
           random_forest                            0  
           tabm                                     0  
           xgboost                                  0

,reference,metric,delta_mean,delta_lower,delta_upper,valid_draws
26,baseline_legacy_strong_inner_l1,macro_year_AP,0.110946,0.038686,0.185600,2000
27,baseline_legacy_strong_inner_l1,macro_year_recall_top5,0.091709,0.009210,0.175560,2000
54,baseline_v1_lags_xgb,macro_year_AP,0.072131,0.002153,0.143583,2000
55,baseline_v1_lags_xgb,macro_year_recall_top5,0.108929,0.028671,0.190816,2000
82,baseline_v1_raw_catboost,macro_year_AP,0.095360,0.029102,0.165932,2000
83,baseline_v1_raw_catboost,macro_year_recall_top5,0.129255,0.047207,0.210886,2000
110,baseline_v1_selected_rf,macro_year_AP,0.087873,0.031687,0.147572,2000
111,baseline_v1_selected_rf,macro_year_recall_top5,0.112987,0.046385,0.183844,2000


,role,groups,AP_mean,AP_min,AP_max,recall5
0,DL,3,0.137654,0.077361,0.206000,0.413171
1,ML,3,0.142885,0.096877,0.206075,0.359648


Company stress ranges are not confidence intervals.


## Reproduction boundary

The final run is frozen. Do not delete confirmation or historical locks. Per-trial configurations, all aggregate trials and source code are retained. Private data, company-level predictions and serialized models are supplied locally by the owner, not downloaded from this repository.

From the repository root, run `scripts/run.sh scripts/verify_project.py` for public evidence checks. Read `docs/REPRODUCTION.md` for fresh registered recipe replication and optional trusted local model checks. This notebook never trains models. Original time windows, incomplete trials and correction records remain under `docs/research_origin/`.